In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "daysim-validation-notebook": ["school_location", "school_tour_mode", "school_trip_mode", "telecommute", "time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import plotly.express as px
import polars as pl
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
hh = util.get_hh_data(summary_config)

hh = hh.\
    join(util.get_parcel_landuse_data(summary_config), how="left", left_on='hhno', right_on='parcelid').\
    join(util.get_parcel_geog(input_config, summary_config), how="left", left_on='hhparcel', right_on='ParcelID')

person = util.get_person_data(summary_config)
tour = util.get_tour_data(summary_config)
parcel_geog = util.get_parcel_geog(input_config, summary_config)

In [4]:
# Group income, hh density, and employment density into 4 groups
model_hh = hh.filter(pl.col('source') == 'model').select(['hhincome', 'emptot_1', 'hh_1'])
var_group = {
    'hhincome': [model_hh.select(pl.col('hhincome').quantile(q)).item() for q in [0.125, 0.25, 0.50, 0.75]],
    'emptot_1': [model_hh.select(pl.col('emptot_1').quantile(q)).item() for q in [0.125, 0.25, 0.50, 0.75]],
    'hh_1': [model_hh.select(pl.col('hh_1').quantile(q)).item() for q in [0.125, 0.25, 0.50, 0.75]],
}

In [5]:
income_bins = var_group['hhincome']
hh_bins = var_group['hh_1']
emp_bins = var_group['emptot_1']

hh = hh.with_columns([
    pl.when(pl.col('hhincome') < income_bins[0]).then(pl.lit('very low'))
      .when(pl.col('hhincome') < income_bins[1]).then(pl.lit('low'))
      .when(pl.col('hhincome') < income_bins[2]).then(pl.lit('medium'))
      .when(pl.col('hhincome') < income_bins[3]).then(pl.lit('medium-high'))
      .otherwise(pl.lit('high')).alias('hhincome_group'),
    pl.when(pl.col('hh_1') < hh_bins[0]).then(pl.lit('very low'))
      .when(pl.col('hh_1') < hh_bins[1]).then(pl.lit('low'))
      .when(pl.col('hh_1') < hh_bins[2]).then(pl.lit('medium'))
      .when(pl.col('hh_1') < hh_bins[3]).then(pl.lit('medium-high'))
      .otherwise(pl.lit('high')).alias('hh_density_group'),
    pl.when(pl.col('emptot_1') < emp_bins[0]).then(pl.lit('very low'))
      .when(pl.col('emptot_1') < emp_bins[1]).then(pl.lit('low'))
      .when(pl.col('emptot_1') < emp_bins[2]).then(pl.lit('medium'))
      .when(pl.col('emptot_1') < emp_bins[3]).then(pl.lit('medium-high'))
      .otherwise(pl.lit('high')).alias('emp_density_group')
])

In [6]:
person = person.join(hh, on=['hhno', 'source'], how='left')

In [7]:
person = person.join(parcel_geog, left_on='pspcl', right_on='ParcelID', how='left', suffix="_school")
person = person.join(parcel_geog, left_on='hhparcel', right_on='ParcelID', how='left', suffix="_home")

In [8]:
person

hhno,pagey,pdiary,pgend,pno,ppaidprk,pproxy,pptyp,psaudist,psautime,psexpfac,pspcl,pstaz,pstyp,ptpass,puwarrp,puwdepp,puwmode,pwaudist,pwautime,pwpcl,pwtaz,pwtyp,source,worker_type,pptyp_label,hh515,hhcu5,hhexpfac,hhftw,hhhsc,hhincome,hhoad,hhparcel,hhptw,hhret,hhsize,…,equity_focus_areas_2023__efa_lep_school,equity_focus_areas_2023__efa_youth_school,equity_focus_areas_2023__efa_older_school,equity_focus_areas_2023__efa_dis_school,BaseYear_school,all_day_transit_school,frequent_transit_school,hct_school,min_transit_school,Unnamed: 0_home,geometry_home,GEOID20_home,Census2020BlockGroup_home,Census2020Tract_home,Census2020Block_home,maz_id_home,rg_proposed_home,CityName_home,CountyName_home,TAZ_home,District_home,district_name_home,GrowthCenterName_home,mic_home,place_name_2020_home,L0ElmerGeo_DBO_tract2020_nowater_geoid20_home,equity_focus_areas_2023__efa_poc_home,equity_focus_areas_2023__efa_pov200_home,equity_focus_areas_2023__efa_lep_home,equity_focus_areas_2023__efa_youth_home,equity_focus_areas_2023__efa_older_home,equity_focus_areas_2023__efa_dis_home,BaseYear_home,all_day_transit_home,frequent_transit_home,hct_home,min_transit_home
i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,i64,str,str,str,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,…,f64,f64,f64,f64,i64,f64,f64,f64,f64,i64,str,f64,f64,f64,f64,i64,str,str,str,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64
230001733,50,0,1,1,0,0,1,-1.0,-1.0,4743.452559,-1,-1,0,0,360,800,-1,10.07,20.91,720927,3625,1,"""survey""","""commuter""","""1: full time worker""",1,0,4743.452559,1,1,175000,0,718203,1,0,4,…,null,null,null,null,null,null,null,null,null,718202,"""POINT (1188546.49888 243428.13…",5.3035e14,5.3035e11,5.3035e10,5.3035e14,37861,"""Core Cities""",null,"""Kitsap""",3533.0,9.0,"""Kitsap""","""Not in RGC""",null,"""Silverdale""",5.3035e10,0.0,1.0,0.0,0.0,2.0,2.0,2023,0.0,0.0,0.0,0.0
230001733,17,0,1,2,0,0,6,5.81,14.23,4743.452559,652459,3644,1,1,-1,-1,-1,-1.0,-1.0,-1,-1,0,"""survey""","""-1""","""6: grade school student/child …",1,0,4743.452559,1,1,175000,0,718203,1,0,4,…,0.0,1.0,1.0,1.0,2023,0.0,0.0,0.0,1.0,718202,"""POINT (1188546.49888 243428.13…",5.3035e14,5.3035e11,5.3035e10,5.3035e14,37861,"""Core Cities""",null,"""Kitsap""",3533.0,9.0,"""Kitsap""","""Not in RGC""",null,"""Silverdale""",5.3035e10,0.0,1.0,0.0,0.0,2.0,2.0,2023,0.0,0.0,0.0,0.0
230001733,14,0,1,3,0,0,7,3.47,9.07,4743.452559,618609,3649,1,1,-1,-1,-1,-1.0,-1.0,-1,-1,0,"""survey""","""-1""","""7: child age 5-15""",1,0,4743.452559,1,1,175000,0,718203,1,0,4,…,0.0,0.0,0.0,0.0,2023,1.0,1.0,1.0,1.0,718202,"""POINT (1188546.49888 243428.13…",5.3035e14,5.3035e11,5.3035e10,5.3035e14,37861,"""Core Cities""",null,"""Kitsap""",3533.0,9.0,"""Kitsap""","""Not in RGC""",null,"""Silverdale""",5.3035e10,0.0,1.0,0.0,0.0,2.0,2.0,2023,0.0,0.0,0.0,0.0
230004663,70,0,2,1,0,0,3,-1.0,-1.0,55.154636,-1,-1,0,0,-1,-1,-1,-1.0,-1.0,-1,-1,0,"""survey""","""-1""","""3: non-worker age 65+""",0,1,36.769757,0,0,62500,0,229260,0,2,3,…,null,null,null,null,null,null,null,null,null,229259,"""POINT (1259338.99196 248204.54…",5.3033e14,5.3033e11,5.3033e10,5.3033e14,34038,"""Metropolitan Cities""","""Seattle""","""King""",242.0,3.0,"""North Seattle-Shoreline""","""Not in RGC""",null,"""Seattle""",5.3033e10,0.0,0.0,0.0,0.0,0.0,0.0,2023,1.0,1.0,1.0,1.0
230004663,70,0,1,2,0,0,3,-1.0,-1.0,55.154636,-1,-1,0,0,-1,-1,-1,-1.0,-1.0,-1,-1,0,"""survey""","""-1""","""3: non-worker age 65+""",0,1,36.769757,0,0,62500,0,229260,0,2,3,…,null,null,null,null,null,null,null,null,null,229259,"""POINT (1259338.99196 248204.54…",5.3033e14,5.3033e11,5.3033e10,5.3033e14,34038,"""Metropolitan Cities""","""Seattle""","""King""",242.0,3.0,"""North Seattle-Shoreline""","""Not in RGC""",null,"""Seattle""",5.3033e10,0.0,0.0,0.0,0.0,0.0,0.0,2023,1.0,1.0,1.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
17360

In [9]:
df_students = person.filter(pl.col('pstyp') > 0)

In [10]:
df_students.filter(pl.col('source') == 'survey').to_pandas().to_clipboard()

In [11]:
df = (
    df_students.filter(pl.col('psaudist') > 0)
    .with_columns((pl.col('psaudist') * pl.col('psexpfac')).alias('wt_dist'))
    .group_by(['source'])
    .agg([
        pl.col('wt_dist').sum().alias('wt_dist'),
        pl.col('psexpfac').sum().alias('psexpfac')
    ])
    .with_columns((pl.col('wt_dist') / pl.col('psexpfac')).alias('Mean Distance to School by Car (psaudist)'))
)
df.select(['source', 'Mean Distance to School by Car (psaudist)'])

source,Mean Distance to School by Car (psaudist)
str,f64
"""survey""",3.963511
"""model""",6.073446


In [12]:
# auto ownership in Income groups
def plot_school_location(df: pl.DataFrame, var: str, title_cat: str, sub_name: str):
    df_plot = (
        df.filter(pl.col('psaudist') > 0)
        .with_columns((pl.col('psaudist') * pl.col('psexpfac')).alias('wt_dist'))
        .group_by(['source', var])
        .agg([
            pl.col('wt_dist').sum().alias('wt_dist'),
            pl.col('psexpfac').sum().alias('psexpfac'),
            pl.col('psexpfac').count().alias('sample count')
        ])
        .with_columns((pl.col('wt_dist') / pl.col('psexpfac')).alias('average_wt_psaudist'))
    )

    fig = px.bar(df_plot.to_pandas(), x=var, y='average_wt_psaudist', color="source",
                 barmode="group", hover_data=['sample count'],
                 title="School Distance by " + title_cat)
    fig.for_each_annotation(lambda a: a.update(text=sub_name + "=<br>" + a.text.split("=")[-1]))
    fig.update_xaxes(title_text=sub_name)
    fig.update_layout(height=400, width=800, font=dict(size=11),
                      yaxis=dict(tickformat=".2f"))
    fig.for_each_yaxis(lambda a: a.update(tickformat=".2f"))
    fig.show()

## School Location by School Geography

In [13]:
plot_school_location(df_students, 'CountyName_school', 'School County', 'County')

In [14]:
plot_school_location(df_students, 'rg_proposed_school', 'Regional Geography', 'Regional Geography')

## School Location by Home Geography

In [15]:
plot_school_location(df_students, 'CountyName_home', 'Home County', 'County')

In [16]:
plot_school_location(df_students, 'rg_proposed_home', 'Home Regional Geography', 'Regional Geography')

## School Location by Person/Household Characteristics

In [17]:
plot_school_location(df_students.filter(pl.col('pptyp').is_in([5, 6, 7])),
                     'pptyp', 'Person Type', 'student type')

In [18]:
plot_school_location(df_students, 'hh_density_group', 'Household Density', 'Household Density Group')

In [19]:
plot_school_location(df_students, 'emp_density_group', 'Employment Density', 'Employment Density Group')

## Distance Bins

In [20]:
# distance to school bins from workplace_location.csv
df_students = df_students.with_columns([
    pl.when(pl.col('psaudist') < 1).then(pl.lit('0-1'))
      .when(pl.col('psaudist') < 2).then(pl.lit('1-2'))
      .when(pl.col('psaudist') < 5).then(pl.lit('2-5'))
      .when(pl.col('psaudist') < 15).then(pl.lit('5-15'))
      .otherwise(pl.lit('15+')).alias('distance_to_school_bin')
])
max_bin = 60
bin_size = 2
df_students = df_students.with_columns(
    pl.when((pl.col('psaudist') >= 0) & (pl.col('psaudist') <= max_bin))
      .then(pl.col('psaudist').truediv(bin_size).floor().mul(bin_size).cast(pl.Int64).cast(pl.Utf8))
      .otherwise(None)
      .alias('d_school_bin_60mi')
)

In [21]:
df_students = df_students.with_columns(
    pl.col('pptyp').replace_strict({5: 'University', 6: 'High School 16+', 7: 'Child Age 5-15'}, default=None).alias('student_type')
)

In [22]:
df_plot = (
    df_students.filter(pl.col('student_type').is_not_null() & pl.col('distance_to_school_bin').is_not_null())
    .group_by(['source', 'distance_to_school_bin'])
    .agg([
        pl.col('psexpfac').sum().alias('psexpfac'),
        pl.col('psexpfac').count().alias('sample count')
    ])
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="distance_to_school_bin", y="percentage", color="source", barmode="group",
             hover_data=['sample count'],
             title="Distance to school")
fig.update_layout(height=400, width=700, font=dict(size=11))
fig.show()

In [23]:
df_plot = (
    df_students.filter(pl.col('student_type').is_not_null() & pl.col('distance_to_school_bin').is_not_null())
    .group_by(['source', 'distance_to_school_bin', 'student_type'])
    .agg([
        pl.col('psexpfac').sum().alias('psexpfac'),
        pl.col('psexpfac').count().alias('sample count')
    ])
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'student_type'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="distance_to_school_bin", y="percentage", color="source", barmode="group",
             facet_col="student_type",
             hover_data=['sample count'],
             title="Distance to school by student type")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=400, width=900, font=dict(size=11))
fig.show()

In [24]:
df_plot = (
    df_students.filter(pl.col('student_type').is_not_null() & pl.col('d_school_bin_60mi').is_not_null())
    .group_by(['source', 'student_type', 'd_school_bin_60mi'])
    .agg(pl.col('psexpfac').sum().alias('psexpfac'))
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'student_type'])).alias('percentage'))
)

fig2 = px.line(df_plot.to_pandas(), x='d_school_bin_60mi', y="percentage", color="source",
               facet_col="student_type", facet_col_wrap=2,
               title="Distance to school by income group")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_layout(height=400, width=700, font=dict(size=11))
fig2.show()

In [25]:
df_plot = (
    df_students.filter(pl.col('distance_to_school_bin').is_not_null())
    .group_by(['source', 'hhincome_group', 'distance_to_school_bin'])
    .agg([
        pl.col('psexpfac').sum().alias('psexpfac'),
        pl.col('psexpfac').count().alias('sample count')
    ])
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'hhincome_group'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="distance_to_school_bin", y="percentage", color="source", barmode="group",
             facet_col="hhincome_group", facet_col_wrap=2,
             hover_data=['sample count'],
             title="Distance to school by household income")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=900, width=700, font=dict(size=11))
fig.show()

In [26]:
df_plot = (
    df_students.filter(pl.col('d_school_bin_60mi').is_not_null())
    .group_by(['source', 'hhincome_group', 'd_school_bin_60mi'])
    .agg(pl.col('psexpfac').sum().alias('psexpfac'))
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'hhincome_group'])).alias('percentage'))
)

fig2 = px.line(df_plot.to_pandas(), x='d_school_bin_60mi', y="percentage", color="source",
               facet_col="hhincome_group", facet_col_wrap=2,
               title="Distance to school by income group")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_layout(height=400, width=700, font=dict(size=11))
fig2.show()

In [27]:
df_plot = (
    df_students.filter(pl.col('distance_to_school_bin').is_not_null())
    .group_by(['source', 'hh_density_group', 'distance_to_school_bin'])
    .agg([
        pl.col('psexpfac').sum().alias('psexpfac'),
        pl.col('psexpfac').count().alias('sample count')
    ])
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'hh_density_group'])).alias('percentage'))
)

fig = px.bar(df_plot.to_pandas(), x="distance_to_school_bin", y="percentage", color="source", barmode="group",
             facet_col="hh_density_group", facet_col_wrap=2,
             hover_data=['sample count'],
             title="Distance to school by household density")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=900, width=700, font=dict(size=11))
fig.show()

In [28]:
df_plot = (
    df_students.filter(pl.col('d_school_bin_60mi').is_not_null())
    .group_by(['source', 'hh_density_group', 'd_school_bin_60mi'])
    .agg(pl.col('psexpfac').sum().alias('psexpfac'))
    .with_columns((100 * pl.col('psexpfac') / pl.col('psexpfac').sum().over(['source', 'hh_density_group'])).alias('percentage'))
)

fig2 = px.line(df_plot.to_pandas(), x='d_school_bin_60mi', y="percentage", color="source",
               facet_col="hh_density_group", facet_col_wrap=2,
               title="Distance to school by household density")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_layout(height=400, width=700, font=dict(size=11))
fig2.show()